# Signal Idea 4: Stochastic-Calculus Inspired Variance Risk Premium Proxy

Research notebook capturing one implementable signal concept from quant literature (non-HFT, not price-only).


In [ ]:
# Setup: imports and robust paths
import sys
from pathlib import Path

import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES_ROOT = [CWD, CWD.parent]

for root in CANDIDATES_ROOT:
    if (root / "config" / "settings.py").exists():
        PROJECT_ROOT = root
        break
else:
    PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def resolve_first_existing(paths: list[Path], fallback: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return fallback

DATA_DIR = resolve_first_existing(
    [root / "data" / "factors" for root in CANDIDATES_ROOT],
    PROJECT_ROOT / "data" / "factors",
)
DUCKDB_PATH = resolve_first_existing(
    [root / "data" / "factors" / "factors.duckdb" for root in CANDIDATES_ROOT],
    DATA_DIR / "factors.duckdb",
)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"DuckDB path  : {DUCKDB_PATH}")
print(f"Data directory exists: {DATA_DIR.exists()}")
print(f"DuckDB exists: {DUCKDB_PATH.exists()}")
if not DUCKDB_PATH.exists():
    print("⚠️ DuckDB file missing. Use parquet files directly for prototype tests.")


In [ ]:
from IPython.display import display
# Quick local data inventory for feasibility
candidates = [
    DATA_DIR / "prices.parquet",
    DATA_DIR / "factors_price.parquet",
    DATA_DIR / "factors_all.parquet",
    DATA_DIR / "macro.parquet",
    DATA_DIR / "macro_z.parquet",
]

rows = []
for p in candidates:
    rows.append({
        "file": p.name,
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / (1024**2), 2) if p.exists() else None,
    })

display_df = pd.DataFrame(rows)
display_df


## What this idea is
Use an approximation of variance risk premium (VRP): expected variance minus realized variance.

Without full options data, start with a proxy:
- Forecast variance via GARCH/EGARCH or HAR-RV style model
- Compare to subsequent realized variance
- Use VRP sign/magnitude as timing signal for equity beta and low-vol sleeves

This is rooted in stochastic volatility frameworks, not raw prices.


## Paper lineage
- Carr and Wu (2009), *Variance Risk Premiums*.
- Bollerslev, Tauchen, Zhou (2009), *Expected Stock Returns and Variance Risk Premia*.

Why relevant: links continuous-time volatility risk pricing to practical timing overlays.


## Feasibility in this project
**Feasibility: Medium**

- Realized variance is easy from local daily returns.
- Full canonical VRP uses implied vol (options), but a proxy can be prototyped immediately.
- Adds value as a regime/tail-risk overlay for existing strategies.


## Data requirements and availability
- **Already local**: daily prices for realized variance.
- **Optional external for stronger version**: VIX term structure (CBOE, Stooq mirrors, Quandl/Nasdaq Data Link).

If options data is missing, proxy VRP remains testable with current data.


## Minimal prototype notebook cell
```python
# pseudo-outline
# 1) Compute rolling realized variance from index returns
# 2) Fit simple variance forecast model
# 3) vrp_proxy = forecast_var - next_realized_var
# 4) Use vrp_proxy quantiles to scale risk exposure
```


## Recommended next experiment in this repo
1. Build a minimal feature table using currently available parquet files.
2. Run a simple walk-forward backtest using existing portfolio metrics.
3. Add one external dataset only if it materially improves explanatory power.
